# Demonstration of Possible Results


# TODO

```py

# create a proper dashboard with ability to use filters like in Wyscout's advanced search

# for the top N (N=15 default) players, give comparative analysis of their stats, such as market value, and potential resale value

# dashboard should be self contained and only require installs on the developer side, the end user should not have to install anything
# The dashboard should be able to run on a local server and be accessible via a web browser.

```


This notebook will be dedicated to the development of an initial template, for demonstration purposes. We were tasked with deveoping a product with a dashboard-like output.

Our goal for this session will be to setup a structured report generation system. In order to do so, we want the following pipeline to be strict and fully functional:

 1. fetching all player data
 2. preserve all identification structural elements, such as country, league and name (for and disambuguation purposes) 
 3. make use of relevant first-pull information (currently manual, API is not applicable nor is webscrape) to define proper relevance metrics
 4. develop techniques to display information in meaningful, comparative plots
 5. define a dashboard-like structure, with filtering capabilities for convenience
 6. add a pdf output functionality to expedite report quality and sharing 

## Frameworks & Libraries

We'll be using standard DS and graph plotting libraries for this demo.

In [9]:
# read ../data/2025-26 folder. This contains multiple subfolders, which can contain csv files. If they do contain csv, use them, flag otherwise.
import re
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

## Getting Working Data

Fetch all `.csv`:

In [10]:
SEASON = "2025-2026"                   
DATA_DIR = Path("../data") / SEASON      

frames, flagged = [], []
for sub in sorted([d for d in DATA_DIR.iterdir() if d.is_dir()]):
    csvs = list(sub.glob("*.csv"))
    if not csvs:

        # flags and skips folders with no csv
        flagged.append(sub.name)         
        continue
    f = pd.read_csv(csvs[0])

    # pulls country code and league from folder name
    m = re.search(r"\[([A-Za-z]{2,3})\]\s*-\s*\((.+)\)", sub.name)   
    f["Country"] = m.group(1) if m else ""
    f["League"]  = m.group(2) if m else sub.name
    frames.append(f)

players = pd.concat(frames, ignore_index=True)
print(f"{len(players)} players across {players['League'].nunique()} leagues")
if flagged:
    print("FLAGGED (no csv, skipped):")
    for name in flagged:
        print("  -", name)

39650 players across 68 leagues
FLAGGED (no csv, skipped):
  - 36 - [CRO] - (First NL)
  - 41 - [USA] - (Major League Soccer)
  - 42 - [USA] - (USL Championship)
  - 60 - [RSA] - (Premier Division)
  - 63 - [ALG] - (Ligue 1)


## Performance & Potential Indicators

<center>

*The core objective of identifying soccer players for profitable resale (the "buy-low, sell-high" trading model)*

</center>

<center>

*relies on **decoupling a player's underlying talent from their current market valuation**.*

</center>

We didn't want to delegate this responsibility to a simple AI chatbot. From the previous reports (given to us as example), we identified clear signs of "drop info and trust output" for report building and reasoning.

In order to get industry certified techniques, we decided to carry out an investigative process and research actual published evidence. This will also allow us to have more robust control of the AI's training process and quality of output, defining a sort of "ground truth" to aspire to reach.

### Satistically Indicative Features for Player Value

According to a study performed by [AMSTATNEWS](https://magazine.amstat.org/) on *"a global dataset of 5,682 professional players and their ratings from FIFA analysts"*, the *"**reactions**, **composure**, **short** **passing**, and **long** **passing** are **significantly correlated with value**, with **technical skills** being **consistently more important than raw physical ability**"*.

#### Feature Correlation & Player Quality Comparison

<center>

![](imgs_demo/correlation.png)
![](imgs_demo/comparison.png)

</center>

### A Study on Performance Metrics as Estimators for Market Value

When considering the study performed by [Yalçınkaya & Işık (2024)](https://doi.org/10.54141/psbd.1489554) in the *Pamukkale Journal of Sport Sciences* on *"1,508 players across Europe's top five leagues, pairing 28 WhoScored performance metrics with Transfermarkt market values"*, the *"**total goals**, **shots per game**, **assists**, **through balls**, **dribbles**, and **key passes** are the **strongest positive correlates of market value**, with **attacking and creative output being consistently more predictive than defensive workload**"*. 

A **Random Forest** model reached an **R² of 0.90**, while **age** carried the **strongest negative correlation (−0.27)**, indicating that younger players are the most preferable.

## Research Conclusions

From the dedicated studies above, we have separated all traits and skills by descending priority, for each role, and **matching them to their Wyscout equivalent** (when possible).

### Role-Independent Features

 1. **Goal output**
    1. Wyscout: combination of `Non-penalty goals per 90`, `Goals per 90`, `xG per 90`
 2. **Shot volume & threat**
    1. Wyscout: `Shots per 90`, `Shots on target, %`
 2. **Chance creation / assists**
    3. Wyscout: `Assists per 90`, `xA per 90`
 3. **Line-breaking passing / through balls**
    1. Wyscout: `Through passes per 90`, `Accurate through passes, %`
 4. **Ball-carrying / dribbling**
    1. Wyscout: `Dribbles per 90`, `Successful dribbles, %`, `Progressive runs per 90`
 5. **Key passing** 
    1. Wyscout: `Key passes per 90`
 6. **Passing volume & accuracy (short & long)
    1. Wyscout: `Passes per 90`, `Accurate passes, %`, `Accurate short / medium passes, %`, `Accurate long passes, %`
 7. **Playing time / reliability**
    1. Wyscout: `Minutes played`, `Matches played`
 8.  **Drawing fouls (being targeted)**
     1.  Wyscout: `Fouls suffered per 90`
 9.  **Age**: considered to be the decoupling variable, with *negative* against current value, *positive* for resale, so it enters the value model and the resale weighting with opposite signs. 
     1.  Wyscout: `Age`

TODO: Role based is giving issues due to underrepresentation!

## Feature Setup: Research-Based Indicators of Potential

In [11]:
# map Wyscout primary position code -> role bucket
ROLE = {}
for p in ["GK"]:                         ROLE[p] = "GK"
for p in ["CB","RCB","LCB"]:             ROLE[p] = "CB"
for p in ["RB","LB","RWB","LWB"]:        ROLE[p] = "FB"
for p in ["DMF","RDMF","LDMF"]:          ROLE[p] = "DM"
for p in ["CMF","RCMF","LCMF"]:          ROLE[p] = "CM"
for p in ["AMF"]:                        ROLE[p] = "AM"
for p in ["LW","RW","LWF","RWF","LAMF","RAMF"]: ROLE[p] = "W"
for p in ["CF"]:                         ROLE[p] = "CF"

# ROLE QUALITY — percentile these WITHIN the role to score "how good at the job"
ROLE_METRICS = {
 "GK": ["Save rate, %", "Prevented goals per 90", "Conceded goals per 90",
        "Exits per 90", "Accurate passes, %", "Accurate long passes, %"],
 "CB": ["Defensive duels won, %", "Aerial duels won, %", "PAdj Interceptions",
        "Shots blocked per 90", "Successful defensive actions per 90",
        "Accurate passes, %", "Accurate progressive passes, %"],
 "FB": ["Defensive duels won, %", "PAdj Interceptions", "Successful defensive actions per 90",
        "Crosses per 90", "Accurate crosses, %", "Progressive runs per 90",
        "xA per 90", "Key passes per 90"],
 "DM": ["PAdj Interceptions", "Defensive duels won, %", "Successful defensive actions per 90",
        "Accurate passes, %", "Accurate forward passes, %", "Progressive passes per 90",
        "Received passes per 90"],
 "CM": ["Progressive passes per 90", "Accurate passes, %", "Key passes per 90",
        "xA per 90", "Progressive runs per 90", "Passes to final third per 90",
        "Duels won, %"],
 "AM": ["xA per 90", "Key passes per 90", "Through passes per 90", "Shot assists per 90",
        "Passes to penalty area per 90", "Dribbles per 90", "Successful dribbles, %",
        "Touches in box per 90", "xG per 90"],
 "W":  ["Successful dribbles, %", "Progressive runs per 90", "Accelerations per 90",
        "xA per 90", "Key passes per 90", "Crosses per 90", "Accurate crosses, %",
        "xG per 90", "Touches in box per 90", "Fouls suffered per 90"],
 "CF": ["Non-penalty goals per 90", "xG per 90", "Goal conversion, %", "Shots on target, %",
        "Touches in box per 90", "Head goals per 90", "Aerial duels won, %",
        "Fouls suffered per 90"],
}

# VALUE DRIVERS — the paper's selected set (Wyscout equivalents). Feed to the value-gap model.
# Ranked within role too, so a CB isn't punished for scoring less than a CF.
VALUE_FEATURES = ["Age", "Goals per 90", "Shots per 90", "Assists per 90", "xA per 90",
                  "Through passes per 90", "Dribbles per 90", "Successful dribbles, %",
                  "Key passes per 90", "Passes per 90", "Accurate passes, %",
                  "Fouls suffered per 90", "Minutes played"]

## Indicator Simulation: Top 15 Players

This simple dashboard demonstration will use a linear, equally weighted combination of all of the research results' most meaningful indicators, to ascertain how effective they are at identifying market value.

This will allow us to:
 1. determine indicator relevance
 2. massively reduce manual download search space (narrows down to N players)


To respect class balance, we'll consider the attacker role: every role-independent feature we retained (goal output, shot threat, assists,
through balls, dribbling, key passes) is measuring **attacking and creative production**, and thus these indicators are best calibrated for this specific population.

TODO: improve aggregation of indicators for score, linear and equal weights is not conceptually correct

Let's start by pulling the best attackers, based on:

 1. `Goals per 90`
 2. `xG per 90`
 3. `Shots per 90`
 4. `Assists per 90`
 5. `xA per 90`
 6. `Through passes per 90`
 7. `Dribbles per 90`
 8. `Successful dribbles, %`
 9. `Progressive runs per 90`
 10. `Key passes per 90`
 11. `Passes per 90`
 12. `Accurate passes, %`
 13. `Fouls suffered per 90`
 14. `Minutes played`

### Indicator Aggregation and Technical Issues

It's important to notice that to combine metrics with different units (per-90 counts, %, minutes) we must first turn each into a **percentile (0 to 1) within the attacking group**. In other words, "what fraction of attackers does this player beat?".

Essentially, the best scoring player will have a `Goals per 90` indicator of 1, and so forth. Then, each player has an averaged result of all columns, resulting in a very rough approximation of a global market value indication. 

The problem with this is that averaging all of these columns would accumulate redundancies, which can be seen in the dribbling case (dribbles per 90, successful dribble %, etc): this would give a larger weight to that skill because it's represented more often.

For this reason, we'll employ a very simple double-average solution:
 1. average the value of all dribbles into 1 column
 2. average all columns, with dribble being represented only once instead of multiple times

***NOTE***: age percentile is inverted because less is better in this case

TODO: probably better fixes then double average

### Comparing Approximated to Real Top 15

TODO: clean and study-explain code better

In [12]:
# primary position code -> role bucket (uses ROLE map)
players = players.copy()
players["Pos"]  = players["Position"].map(lambda p: "NA" if pd.isna(p) else str(p).split(",")[0].strip())
players["Role"] = players["Pos"].map(lambda p: ROLE.get(p, "MID"))

# best-represented class = attacking group
att = players[players["Role"].isin(["AM", "W", "CF"])].copy()

# the 14 indicators, grouped into 10 skills so each skill weighs equally (10%)
CONCEPTS = {
    "Goal output":     ["Goals per 90", "xG per 90"],
    "Shot threat":     ["Shots per 90"],
    "Chance creation": ["Assists per 90", "xA per 90"],
    "Through balls":   ["Through passes per 90"],
    "Dribbling":       ["Dribbles per 90", "Successful dribbles, %", "Progressive runs per 90"],
    "Key passing":     ["Key passes per 90"],
    "Passing":         ["Passes per 90", "Accurate passes, %"],
    "Fouls drawn":     ["Fouls suffered per 90"],
    "Minutes":         ["Minutes played"],
    "Age (younger)":   None,   # inverted, handled separately
}

# stage 1: every metric -> percentile within the attacking group
metric_cols = [col for cols in CONCEPTS.values() if cols for col in cols]
percentile  = att[metric_cols].apply(pd.to_numeric, errors="coerce").rank(pct=True)

# stage 2a: average columns within each skill -> one score per skill
skill_scores = {
    name: (percentile[cols].mean(axis=1) if cols
           else 1 - pd.to_numeric(att["Age"], errors="coerce").rank(pct=True))
    for name, cols in CONCEPTS.items()
}

# stage 2b: average the 10 skills -> each skill = 10%
att["Value index"] = (pd.DataFrame(skill_scores).fillna(0).mean(axis=1) * 100).round(1)

# rank by our index (assumed) and by real price (actual)
att["Assumed rank"] = att["Value index"].rank(ascending=False, method="min").astype(int)
att["Actual rank"]  = pd.to_numeric(att["Market value"], errors="coerce").rank(ascending=False, method="min").astype(int)

print(f"{len(att)} attackers | {len(CONCEPTS)} skills, {100/len(CONCEPTS):.0f}% each")

15012 attackers | 10 skills, 10% each


Let's identify the Top 15 potentially relevant players by Indicator Aggregation:

In [13]:
SHOW = ["Player", "League", "Position", "Age", "Market value", "Value index", "Assumed rank", "Actual rank"]
best_by_indicator = att.sort_values("Value index", ascending=False).head(15)[SHOW].reset_index(drop=True)
best_by_indicator

,Player,League,Position,Age,Market value,Value index,Assumed rank,Actual rank
0,J. Yeboah,Serie B,CF,26,2500000,86.2,1,299
1,I. Ferrah,Eerste Divisie,"RWF, RAMF, RW",20,0,86.0,2,6505
2,Helinho,Liga MX,"RW, RAMF, CF",26,7000000,85.6,3,59
3,K. Alajbegović,Bundesliga,"LAMF, LW, AMF",18,0,85.5,4,6505
4,G. Diakité,Super League,"CF, LCMF, AMF",20,0,85.0,5,6505
5,E. Otoo,A.LeCoq Premium Liiga,"LAMF, LW",22,0,83.4,6,6505
6,Cryzan,CSL,"CF, AMF, LAMF",30,4500000,83.3,7,148
7,K. Karetsas,Pro League,"AMF, RAMF, RWF",18,0,83.2,8,6505
8,M. Godts,Eredivisie,"LWF, LAMF",21,0,83.1,9,6505
9,C. Tzolis,Pro League,LAMF,24,10000000,83.0,10,21


Next, we'll fetch the actual Top 15, based on market value:

In [14]:
best_by_market = att.sort_values("Market value", ascending=False).head(15)[SHOW].reset_index(drop=True)
best_by_market

,Player,League,Position,Age,Market value,Value index,Assumed rank,Actual rank
0,R. Sterling,Eredivisie,"LWF, LAMF",31,35000000,48.7,7593,1
1,T. Abraham,Süper Lig,CF,28,30000000,51.6,6380,2
2,A. Mitrović,Qatar Stars League,CF,31,28000000,51.5,6420,3
3,E. Touré,Süper Lig,"LAMF, LW, CF",24,20000000,59.6,3223,4
4,N. Zaniolo,Süper Lig,"RAMF, RW, AMF",27,20000000,61.5,2588,4
5,A. Colpani,Serie B,"AMF, RW",27,18000000,64.4,1764,6
6,A. Skov Olsen,Premiership,"RAMF, RW",26,18000000,60.4,2945,6
7,N. Lang,Süper Lig,"LAMF, LW",27,18000000,66.8,1219,6
8,A. Saint-Maximin,Liga MX,"LW, LAMF, AMF",29,18000000,72.8,387,6
9,A. Harit,Süper Lig,LAMF,29,15000000,65.4,1524,10


### Results

In [16]:
rho = att["Value index"].corr(pd.to_numeric(att["Market value"], errors="coerce"), method="spearman")
print(f"Spearman(index, market value) = {rho:.3f}   (1.0 = identical ranking, 0 = unrelated)")

comparison = pd.DataFrame({
    "Rank": range(1, 16),
    "Assumed (by indicators)": best_by_indicator["Player"].values,
    "Value idx": best_by_indicator["Value index"].values,
    "assumed €m": (pd.to_numeric(best_by_indicator["Market value"], errors="coerce") / 1e6).round(1).values,
    "its actual €-rank": best_by_indicator["Actual rank"].values,
    "Actual (by market value)": best_by_market["Player"].values,
    "€m": (pd.to_numeric(best_by_market["Market value"], errors="coerce") / 1e6).round(1).values,
})
comparison

Spearman(index, market value) = 0.182   (1.0 = identical ranking, 0 = unrelated)


,Rank,Assumed (by indicators),Value idx,assumed €m,its actual €-rank,Actual (by market value),€m
0,1,J. Yeboah,86.2,2.5,299,R. Sterling,35.0
1,2,I. Ferrah,86.0,0.0,6505,T. Abraham,30.0
2,3,Helinho,85.6,7.0,59,A. Mitrović,28.0
3,4,K. Alajbegović,85.5,0.0,6505,E. Touré,20.0
4,5,G. Diakité,85.0,0.0,6505,N. Zaniolo,20.0
5,6,E. Otoo,83.4,0.0,6505,A. Colpani,18.0
6,7,Cryzan,83.3,4.5,148,A. Skov Olsen,18.0
7,8,K. Karetsas,83.2,0.0,6505,N. Lang,18.0
8,9,M. Godts,83.1,0.0,6505,A. Saint-Maximin,18.0
9,10,C. Tzolis,83.0,10.0,21,A. Harit,15.0


TODO: 
1. check if it's justified that these assumptions are completely wrong or if I've messed up my rationale somewhere, results are quite unsatisfactory
2. how is it assuming players worth 0 so highly? possible that 20 assists in bundesliga 2 are far more valuable then 500 assist in real das codeas... maybe for consistency let's compare players only inside the same league, or between two similar-level leagues